## YOLO Transfer Learning for Human Classification

In [2]:
!pip install ultralytics torch torchvision numpy pandas opencv-python matplotlib

In [3]:
# Импорт всех необходимых модулей
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
import ast
from PIL import Image
import io
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from google.colab import drive

# Подключение Google Drive
drive.mount('/content/drive')

# Загрузка данных
csv_path = '/content/drive/MyDrive/Colab Notebooks/Transfer_YOLO/train.csv'
df = pd.read_csv(csv_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
def convert_image(image_str):
    try:
        # Преобразуем строку в словарь
        image_dict = ast.literal_eval(image_str)

        # Получаем бинарные данные
        img_bytes = image_dict['bytes']

        # Конвертируем в изображение
        img = Image.open(io.BytesIO(img_bytes))
        img = img.convert('RGB')
        return img
    except Exception as e:
        print(f"Ошибка конвертации: {str(e)}")
        return None

# Обработка данных
images = []
labels = []
failed_count = 0

for index, row in df.iterrows():
    img = convert_image(row['image'])
    if img is not None:
        # Ресайз и конвертация в numpy array
        img = img.resize((640, 640))
        img_array = np.array(img) / 255.0  # Нормализация

        images.append(img_array)
        labels.append(row['label'])
    else:
        failed_count += 1

print(f"\nУспешно обработано: {len(images)}/{len(df)} изображений")
print(f"Не удалось обработать: {failed_count} изображений")

# Визуализация примера
plt.figure(figsize=(8, 8))
plt.imshow(images[0])
plt.title(f"Label: {labels[0]}")
plt.axis('off')
plt.show()

NameError: name 'df' is not defined

In [ ]:
train_images, test_images, train_labels, test_labels = train_test_split(
    images, labels, test_size=0.2, random_state=42)

# Преобразование в тензоры PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_images_tensor = torch.tensor(np.array(train_images)).permute(0, 3, 1, 2).float().to(device)
train_labels_tensor = torch.tensor(train_labels).float().unsqueeze(1).to(device)
test_images_tensor = torch.tensor(np.array(test_images)).permute(0, 3, 1, 2).float().to(device)
test_labels_tensor = torch.tensor(test_labels).float().unsqueeze(1).to(device)

# Создание DataLoader
train_dataset = TensorDataset(train_images_tensor, train_labels_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataset = TensorDataset(test_images_tensor, test_labels_tensor)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Загрузка YOLO модели
model = YOLO('yolov8n-seg.pt')
yolo_base = model.model.model

In [ ]:
for param in yolo_base.parameters():
    param.requires_grad = False

# Создание классификационной модели
class HumanClassifier(nn.Module):
    def __init__(self, yolo_base):
        super(HumanClassifier, self).__init__()
        self.yolo_features = yolo_base
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.yolo_features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# Инициализация модели
model = HumanClassifier(yolo_base).to(device)

# Настройка обучения
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# Обучение модели
num_epochs = 10
best_accuracy = 0

In [ ]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        predicted = (outputs > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_accuracy = 100 * correct_train / total_train

    # Валидация
    model.eval()
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            predicted = (outputs > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_accuracy = 100 * correct_val / total_val

    # Сохраняем лучшую модель
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        torch.save(model.state_dict(), 'best_model.pth')

    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {running_loss/len(train_loader):.4f}, '
          f'Train Acc: {train_accuracy:.2f}%, '
          f'Val Acc: {val_accuracy:.2f}%')

print(f'Лучшая точность на валидации: {best_accuracy:.2f}%')

In [ ]:
dummy_input = torch.randn(1, 3, 640, 640).to(device)
torch.onnx.export(model, dummy_input, "human_classifier.onnx")

In [ ]:
!pip install onnx-tf

In [ ]:
import onnx
from onnx_tf.backend import prepare

onnx_model = onnx.load("human_classifier.onnx")
tf_rep = prepare(onnx_model)

converter = tf.lite.TFLiteConverter.from_saved_model(tf_rep.tf_module)
tflite_model = converter.convert()

with open('detect.tflite', 'wb') as f:
    f.write(tflite_model)

print("Модель успешно сохранена как detect.tflite")

In [ ]:
interpreter = tf.lite.Interpreter(model_path='detect.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

correct = 0
total = len(test_images)

for i in range(total):
    input_data = test_images_tensor[i].unsqueeze(0).cpu().numpy()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])

    pred = 1 if output_data > 0.5 else 0
    if pred == test_labels[i]:
        correct += 1

accuracy = 100 * correct / total
print(f'Точность TFLite модели на тестовых данных: {accuracy:.2f}%')


In [ ]:
from google.colab import files
files.download('detect.tflite')